# Illustration Exercise — Introduction

## Context

TF-IDF (Term Frequency–Inverse Document Frequency) measures how discriminating a word is for a specific document relative to a corpus. This exercise fetches recent news headlines for three major financial firms — AAPL, MSFT, JPM — via the `yfinance` API and visualises the top-20 TF-IDF terms as a heatmap. High-weight cells reveal company-specific vocabulary: terms that appear frequently in one firm's coverage but rarely across the whole corpus.

## Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scienceplots
import yfinance as yf
from sklearn.feature_extraction.text import TfidfVectorizer
import pathlib, warnings
warnings.filterwarnings("ignore")

plt.style.use(["science", "no-latex"])

def get_headlines(sym, n=12):
    ticker = yf.Ticker(sym)
    news = ticker.news or []
    out = []
    for item in news[:n]:
        title = (item.get("title") or
                 item.get("content", {}).get("title") or "")
        if title:
            out.append(title)
    return out

tickers = ["AAPL", "MSFT", "JPM"]
corpus, labels = [], []
for t in tickers:
    for h in get_headlines(t):
        corpus.append(h)
        labels.append(t)

print(f"Fetched {len(corpus)} headlines across {len(tickers)} tickers")

## Figure

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english", max_features=20)
X = vectorizer.fit_transform(corpus)
terms = vectorizer.get_feature_names_out()

df = pd.DataFrame(X.toarray(), columns=terms)
df["ticker"] = labels
agg = df.groupby("ticker").mean()

fig, ax = plt.subplots(figsize=(10, 3))
im = ax.imshow(agg.values, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(terms)))
ax.set_xticklabels(terms, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(agg.index)))
ax.set_yticklabels(agg.index)
ax.set_title("Mean TF-IDF Scores Across News Headlines")
plt.colorbar(im, ax=ax, label="TF-IDF weight")
fig.tight_layout()

out_path = pathlib.Path("../../../book/chapters/01-intro/figures/fig_illustration.pdf")
out_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_path, bbox_inches="tight", dpi=150)
print(f"Saved to {out_path.resolve()}")
plt.show()

## Your Turn

Add a fourth ticker of your choice (e.g., `"TSLA"` or `"GS"`), regenerate the heatmap, and describe in one paragraph which terms are unique to that company and what they reveal about its news coverage relative to the other three firms.

# Exercises — Lecture 1: Introduction to LLMs in Finance

See `course/lectures/01-intro/exercises.md` for the problem statements.

**Exercise 1.1 [B]:** Bag-of-Words and TF-IDF on earnings call sentences.

**Exercise 1.2 [I]:** PCA visualisation of financial word embeddings.

**Exercise 1.3 [A]:** Self-attention from scratch in NumPy.

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt

## Exercise 1.1 [B]

[Your solution here]

## Exercise 1.2 [I]

[Your solution here]

## Exercise 1.3 [A]

[Your solution here]

## Data Lab — SEC EDGAR

The [SEC EDGAR REST API](https://data.sec.gov) provides free programmatic access to all public filings. No API key is required — you only need to set a descriptive `User-Agent` header as required by the SEC's fair-access policy. The helper below enforces a 110 ms sleep between requests to stay within the 10 req/s rate limit.

In [ ]:
import requests, time, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

EDGAR_HEADERS = {"User-Agent": "LLM-Finance-Course instructor@dauphine.eu"}

def edgar_get(url):
    time.sleep(0.11)          # SEC rate limit: <=10 req/s
    r = requests.get(url, headers=EDGAR_HEADERS, timeout=15)
    r.raise_for_status()
    return r.json()

def get_cik(ticker):
    data = edgar_get("https://www.sec.gov/files/company_tickers.json")
    for v in data.values():
        if v["ticker"].upper() == ticker.upper():
            return str(v["cik_str"]).zfill(10)
    raise ValueError(f"Ticker {ticker} not found")

def get_submissions(cik):
    return edgar_get(f"https://data.sec.gov/submissions/CIK{cik}.json")

def get_concept(cik, concept):
    return edgar_get(
        f"https://data.sec.gov/api/xbrl/companyconcept/CIK{cik}/us-gaap/{concept}.json")

def get_annual_series(cik, concept):
    """Return a tidy DataFrame of annual (10-K) values for one XBRL concept."""
    data = get_concept(cik, concept)
    usd  = data.get("units", {}).get("USD", [])
    rows = [u for u in usd if u.get("form") == "10-K" and u.get("filed")]
    if not rows:
        for unit_key, vals in data.get("units", {}).items():
            rows = [u for u in vals if u.get("form") == "10-K" and u.get("filed")]
            if rows: break
    df = (pd.DataFrame(rows)[["end","val","filed","accn"]]
            .rename(columns={"end":"date","val":concept})
            .drop_duplicates("date").sort_values("date"))
    df["date"] = pd.to_datetime(df["date"])
    return df

### Exercise [B]: EDGAR Filing Browser

Retrieve the full filing history for Apple Inc. (`AAPL`), summarise by form type, and identify the most recent 10-K.

In [ ]:
# --- Exercise [B]: EDGAR Filing Browser ---
cik  = get_cik("AAPL")
subs = get_submissions(cik)
f    = subs["filings"]["recent"]

# (a) Table of 20 most recent filings
recent = pd.DataFrame({
    "form":   f["form"][:20],
    "date":   f["filingDate"][:20],
    "accn":   f["accessionNumber"][:20],
})
print(recent.to_string(index=False))

# (b) Filing-type frequency bar chart
all_forms = pd.Series(f["form"]).value_counts().head(10)
fig, ax = plt.subplots(figsize=(8, 4))
all_forms.plot.bar(ax=ax, color="steelblue")
ax.set_xlabel("Form type"); ax.set_ylabel("Count")
ax.set_title(f"AAPL filings by type (total = {len(f['form'])})")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

# (c) Most recent 10-K
idx_10k = next(i for i, x in enumerate(f["form"]) if x == "10-K")
print(f"\nMost recent 10-K: {f['filingDate'][idx_10k]}  ({f['accessionNumber'][idx_10k]})")

### Exercise [I]: MD&A Keyword Landscape

Download the most recent 10-K HTML, strip tags, extract the MD&A section, and compute TF-IDF keywords.

In [ ]:
# --- Exercise [I]: MD&A Keyword Landscape ---
STOP = {"the","a","an","and","or","of","in","to","is","are","was","were","for",
        "on","at","by","as","with","this","that","from","it","its","be","been",
        "has","have","had","will","we","our","their","which","not","but","also",
        "more","than","such","may","can","any","all","one","year","million","billion"}

def fetch_10k_html(ticker):
    cik  = get_cik(ticker)
    subs = get_submissions(cik)
    f    = subs["filings"]["recent"]
    idx  = next(i for i, x in enumerate(f["form"]) if x == "10-K")
    acc  = f["accessionNumber"][idx].replace("-", "")
    url  = (f"https://www.sec.gov/Archives/edgar/data/{cik.lstrip('0')}"
            f"/{acc}/{f['primaryDocument'][idx]}")
    time.sleep(0.15)
    return requests.get(url, headers=EDGAR_HEADERS, timeout=30).text

def extract_mda(html):
    text = re.sub(r"<[^>]+>", " ", html)
    m    = re.search(
        r"Item\s+7[.\s]+Management.{0,80}Discussion.*?(?=Item\s+7A|Item\s+8)",
        text, re.IGNORECASE | re.DOTALL)
    raw  = m.group(0) if m else text[:30000]
    return re.sub(r"\s+", " ", raw).strip()

html = fetch_10k_html("AAPL")
mda  = extract_mda(html)
print(f"MD&A word count: {len(mda.split()):,}")

# TF-IDF over sentences
from math import log
sents    = [s.strip() for s in re.split(r"(?<=[.!?])\s+", mda) if len(s.split()) > 4]
tokenize = lambda s: [w.lower() for w in re.findall(r"[a-z]{3,}", s)]
tok_sents = [tokenize(s) for s in sents]
N = len(sents)
df_count = {}
for ts in tok_sents:
    for w in set(ts):
        df_count[w] = df_count.get(w, 0) + 1

word_scores = {}
for ts in tok_sents:
    n = len(ts)
    for w in set(ts):
        tf  = ts.count(w) / n
        idf = log((N + 1) / (df_count.get(w, 1) + 1))
        word_scores[w] = word_scores.get(w, 0) + tf * idf

# With and without stop words
for label, exclude in [("with stop words", set()), ("without stop words", STOP)]:
    top = sorted(
        [(w, s) for w, s in word_scores.items() if w not in exclude],
        key=lambda x: -x[1])[:20]
    words_plot, scores_plot = zip(*top)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(range(20), scores_plot[::-1], color="steelblue")
    ax.set_yticks(range(20)); ax.set_yticklabels(words_plot[::-1])
    ax.set_xlabel("TF-IDF score"); ax.set_title(f"AAPL MD&A top-20 keywords ({label})")
    plt.tight_layout(); plt.show()

### Exercise [A]: Cross-Sector Vocabulary Overlap

Compare MD&A vocabularies across five companies from three sectors and visualise pairwise Jaccard similarity.

In [ ]:
# --- Exercise [A]: Cross-Sector Vocabulary Overlap ---
TICKERS_A = ["AAPL", "MSFT", "JPM", "BAC", "XOM"]

vocabs = {}
for t in TICKERS_A:
    try:
        html_t = fetch_10k_html(t)
        mda_t  = extract_mda(html_t)
        vocabs[t] = set(re.findall(r"[a-z]{3,}", mda_t.lower())) - STOP
        print(f"{t}: {len(vocabs[t]):,} unique tokens")
    except Exception as e:
        print(f"{t}: ERROR — {e}")

n = len(TICKERS_A)
J = np.zeros((n, n))
for i, ti in enumerate(TICKERS_A):
    for j, tj in enumerate(TICKERS_A):
        if ti in vocabs and tj in vocabs:
            inter = len(vocabs[ti] & vocabs[tj])
            union = len(vocabs[ti] | vocabs[tj])
            J[i, j] = inter / union if union else 0

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(J, cmap="YlOrRd", vmin=0, vmax=1)
ax.set_xticks(range(n)); ax.set_xticklabels(TICKERS_A)
ax.set_yticks(range(n)); ax.set_yticklabels(TICKERS_A)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f"{J[i,j]:.2f}", ha="center", va="center", fontsize=9)
plt.colorbar(im, ax=ax, label="Jaccard similarity")
ax.set_title("MD&A Vocabulary Overlap — Jaccard Similarity")
plt.tight_layout(); plt.show()